# Пакетная предобработка архивных документов
Автоматическая обработка всех файлов из папки `data` с сохранением результатов в `preproc_results`.

Пайплайн: выравнивание → разделение на страницы → нормализация освещения

Включает tqdm для отслеживания прогресса обработки.

## 1. Импорт зависимостей

In [ ]:
# !pip install opencv-python-headless numpy matplotlib scikit-image tqdm

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import os

print('✓ Зависимости загружены')

## 2. Конфигурация путей и параметры

In [ ]:
# ─── Пути ──────────────────────────────────────────────────────────────────
DATA_DIR    = Path('data')           # Папка с исходными файлами
OUTPUT_DIR  = Path('preproc_results')  # Папка для сохранения результатов
OUTPUT_DIR.mkdir(exist_ok=True)

# Поддерживаемые расширения
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

# ─── Параметры обработки ────────────────────────────────────────────────────
DESKEW_MAX_ANGLE    = 10    # максимальный угол наклона для поиска (градусы)
SPINE_DARK_THRESH   = 90    # порог яркости для определения полосы переплёта
SPINE_MIN_WIDTH     = 20    # минимальная ширина полосы переплёта (пикселей)
CLAHE_CLIP          = 2.0   # clip limit для CLAHE
CLAHE_TILE          = 16    # размер тайла CLAHE

print(f'📁 Data folder: {DATA_DIR.resolve()}')
print(f'📁 Output folder: {OUTPUT_DIR.resolve()}')
print(f'✓ Конфигурация загружена')

## 3. Вспомогательные функции

In [ ]:
def find_skew_angle(gray: np.ndarray, max_angle: float = 10.0) -> float:
    """
    Определяет угол наклона документа через Hough-трансформ.
    Возвращает угол в градусах (положительный = по часовой стрелке).
    """
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255,
                               cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 1))
    dilated = cv2.dilate(binary, kernel)

    lines = cv2.HoughLinesP(
        dilated,
        rho=1, theta=np.pi / 180,
        threshold=100,
        minLineLength=gray.shape[1] // 4,
        maxLineGap=20
    )

    if lines is None:
        return 0.0

    angles = []
    for x1, y1, x2, y2 in lines[:, 0]:
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if abs(angle) <= max_angle:
            angles.append(angle)

    if not angles:
        return 0.0

    return float(np.median(angles))


def rotate_image(img: np.ndarray, angle: float) -> np.ndarray:
    """Поворачивает изображение на angle градусов относительно центра."""
    h, w = img.shape[:2]
    cx, cy = w // 2, h // 2
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)

    cos, sin = abs(M[0, 0]), abs(M[0, 1])
    new_w = int(h * sin + w * cos)
    new_h = int(h * cos + w * sin)
    M[0, 2] += new_w / 2 - cx
    M[1, 2] += new_h / 2 - cy

    fill = (255, 255, 255) if img.ndim == 3 else 255
    return cv2.warpAffine(img, M, (new_w, new_h),
                          flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT,
                          borderValue=fill)


def find_spine(gray: np.ndarray,
               dark_thresh: int = 80,
               min_width: int = 20,
               search_fraction: float = 0.5) -> int:
    """
    Находит центр полосы переплёта.
    Ищет в центральной search_fraction части изображения.
    Возвращает x-координату центра переплёта.
    """
    h, w = gray.shape
    margin_v = h // 10
    roi = gray[margin_v: h - margin_v, :]

    profile = roi.mean(axis=0)

    cx = w // 2
    half = int(w * search_fraction / 2)
    search = profile[cx - half: cx + half]
    offset = cx - half

    dark_mask = search < dark_thresh

    if not dark_mask.any():
        return w // 2

    best_start, best_len = 0, 0
    cur_start, cur_len = 0, 0
    for i, v in enumerate(dark_mask):
        if v:
            if cur_len == 0:
                cur_start = i
            cur_len += 1
            if cur_len > best_len:
                best_len, best_start = cur_len, cur_start
        else:
            cur_len = 0

    if best_len < min_width:
        return w // 2

    return offset + best_start + best_len // 2


def normalize_lighting(img_rgb: np.ndarray,
                       clip_limit: float = 2.0,
                       tile_size: int = 16) -> np.ndarray:
    """
    Нормализует освещение через CLAHE в пространстве LAB.
    Работает только с каналом L (яркость), цвет не трогает.
    """
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=clip_limit,
        tileGridSize=(tile_size, tile_size)
    )
    L_eq = clahe.apply(L)

    lab_eq = cv2.merge([L_eq, a, b])
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)


def save_image(img_rgb: np.ndarray, path: Path) -> bool:
    """Сохраняет требование в格式 BGR (для OpenCV) и возвращает успех."""
    try:
        cv2.imwrite(str(path), cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR))
        return True
    except Exception as e:
        print(f'  ⚠ Ошибка сохранения {path}: {e}')
        return False

print('✓ Функции обработки загружены')

## 4. Функция обработки одного файла

In [ ]:
def preprocess_file(input_path: Path) -> dict:
    """
    Обрабатывает один файл через полный пайплайн:
    1. Загрузка
    2. Выравнивание (deskew)
    3. Поиск и разделение переплета
    4. Нормализация освещения
    5. Сохранение результатов
    
    Возвращает словарь со статусом и сведениями об обработке.
    """
    result = {
        'file': input_path.name,
        'status': 'error',
        'message': '',
        'files_saved': []
    }

    try:
        # Загрузка
        img_bgr = cv2.imread(str(input_path))
        if img_bgr is None:
            result['message'] = 'Не удалось прочитать файл'
            return result

        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        # Выравнивание
        angle = find_skew_angle(img_gray, max_angle=DESKEW_MAX_ANGLE)
        img_deskewed_rgb = rotate_image(img_rgb, angle)
        img_deskewed_gray = rotate_image(img_gray, angle)

        # Поиск переплета и разделение
        spine_x = find_spine(img_deskewed_gray,
                             dark_thresh=SPINE_DARK_THRESH,
                             min_width=SPINE_MIN_WIDTH)

        page_left_rgb = img_deskewed_rgb[:, :spine_x]
        page_right_rgb = img_deskewed_rgb[:, spine_x:]

        # Нормализация освещения
        left_norm = normalize_lighting(page_left_rgb, CLAHE_CLIP, CLAHE_TILE)
        right_norm = normalize_lighting(page_right_rgb, CLAHE_CLIP, CLAHE_TILE)

        # Сохранение
        stem = input_path.stem
        left_path = OUTPUT_DIR / f'{stem}_left.jpg'
        right_path = OUTPUT_DIR / f'{stem}_right.jpg'

        if save_image(left_norm, left_path):
            result['files_saved'].append(left_path.name)
        if save_image(right_norm, right_path):
            result['files_saved'].append(right_path.name)

        result['status'] = 'success'
        result['message'] = f'✓ Обработано, сохранено {len(result["files_saved"])} файлов'

    except Exception as e:
        result['message'] = f'Исключение: {str(e)}'

    return result

print('✓ Функция обработки файла готова')

## 5. Пакетная обработка файлов с прогресс-баром (tqdm)

In [ ]:
# Получаем список всех файлов для обработки
input_files = [
    f for f in DATA_DIR.iterdir()
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS
]

if not input_files:
    print(f'⚠ В папке {DATA_DIR} не найдено файлов изображений!')
else:
    print(f'📊 Найдено файлов для обработки: {len(input_files)}')
    print()

    # Статистика
    successful = 0
    failed = 0
    total_files_saved = 0

    # Обработка с прогресс-баром tqdm
    for input_path in tqdm(input_files, desc='Обработка', unit='файл', colour='green'):
        result = preprocess_file(input_path)

        if result['status'] == 'success':
            successful += 1
            total_files_saved += len(result['files_saved'])
        else:
            failed += 1

    print()
    print('=' * 60)
    print('📈 РЕЗУЛЬТАТЫ ПАКЕТНОЙ ОБРАБОТКИ')
    print('=' * 60)
    print(f'✓ Успешно обработано: {successful}')
    print(f'✗ Ошибок: {failed}')
    print(f'📁 Всего файлов сохранено: {total_files_saved}')
    print(f'📂 Выходная папка: {OUTPUT_DIR.resolve()}')
    print('=' * 60)

## 6. Проверка результатов

In [ ]:
# Список всех сохраненных файлов
output_files = list(OUTPUT_DIR.glob('*'))

print(f'📁 Содержимое папки {OUTPUT_DIR.name}:')
print()

if output_files:
    for i, f in enumerate(sorted(output_files), 1):
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  {i:2d}. {f.name:50s} ({size_mb:.2f} MB)')
    print()
    print(f'💾 Всего файлов в выходной папке: {len(output_files)}')
else:
    print('  (папка пуста)')

print()
print('✅ Пакетная обработка завершена!')

## 7. Примеры обработанных изображений (опционально)

In [ ]:
# Показать примеры обработанных файлов (первые 4)
output_images = sorted([f for f in OUTPUT_DIR.glob('*.jpg') if f.is_file()])[:4]

if output_images:
    fig, axes = plt.subplots(len(output_images), 1, figsize=(14, 4 * len(output_images)))
    if len(output_images) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, output_images):
        img = cv2.imread(str(img_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
            ax.set_title(f'{img_path.name} — {img.shape[1]}×{img.shape[0]} px', fontsize=11)
            ax.axis('off')

    plt.tight_layout()
    plt.show()
    print('✓ Примеры отображены')
else:
    print('Нет обработанных файлов для отображения')